# Genie360 — 00: Setup Infrastructure

One-time setup notebook:
- Creates Lakebase PostgreSQL tables (`genie_spaces`, `query_analysis`, `rewrite_history`, `injection_log`)
- Grants service principal permissions to `system.query.history`
- Configures SQL Warehouse connection
- Validates connectivity

In [ ]:
from __future__ import annotations

import os
import re
import json
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter
from typing import Any

import yaml
import pandas as pd
from dotenv import load_dotenv

def _find_file(filename: str) -> Path:
    """Search cwd, parent, and grandparent for *filename*."""
    for directory in [Path(os.getcwd()), Path(os.getcwd()).parent, Path(os.getcwd()).parent.parent]:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"{filename} not found in cwd or ancestor directories")


def load_yaml_config(config_path: str | Path | None = None) -> dict[str, Any]:
    """Load application config from *config.yml* and credentials from *.env*.

    Non-secret tunables (catalog, schema, tables, etc.) live in config.yml.
    Credentials (DATABRICKS_HOST / DATABRICKS_TOKEN) are loaded from .env.
    """
    if config_path is None:
        config_path = _find_file("config.yml")
    config_path = Path(config_path)

    with open(config_path) as fh:
        cfg = yaml.safe_load(fh)
    print(f"✅ Loaded config from {config_path}")

    return cfg

In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from genie360.pipeline import load_genie360_config

# config = load_genie360_config(project_root / "genie360" / "config.yml")

config = load_yaml_config()
print(f"Config loaded: {list(config.keys())}")

✅ Loaded config from /Users/sourav.banerjee/Documents/Codebases/Databricks Projects/AutoGenie/genie360/config.yml
Config loaded: ['warehouse_id', 'cluster_id', 'llm_endpoint', 'llm_max_tokens', 'llm_temperature', 'lookback_days', 'max_queries_per_space', 'severity_escalation', 'safe_limit', 'date_range_hint_days', 'cardinality_threshold', 'min_confidence_for_injection', 'explain_plan_comparison', 'auto_inject_enabled', 'max_rewrites_per_injection', 'cost_per_dbu', 'warehouse_size_defaults', 'output_path']


In [6]:
# ── Get Spark Session ────────────────────────────────────────────────────────
try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.0.0


## 1. Verify System Tables Access

In [7]:
# Verify access to system.query.history
try:
    count = spark.sql("""
        SELECT COUNT(*) as total
        FROM system.query.history
        WHERE start_time >= CURRENT_TIMESTAMP - INTERVAL 1 DAY
    """).collect()[0]["total"]
    print(f"✅ system.query.history accessible — {count} queries in last 24h")
except Exception as e:
    print(f"❌ Cannot access system.query.history: {e}")

✅ system.query.history accessible — 309691 queries in last 24h


In [8]:
# Check for Genie-generated queries
try:
    genie_count = spark.sql("""
        SELECT COUNT(*) as total
        FROM system.query.history
        WHERE start_time >= CURRENT_TIMESTAMP - INTERVAL 30 DAYS
          AND query_source.genie_space_id IS NOT NULL
    """).collect()[0]["total"]
    print(f"✅ Found {genie_count} Genie-generated queries in last 30 days")
except Exception as e:
    print(f"❌ Error querying Genie queries: {e}")

✅ Found 184099 Genie-generated queries in last 30 days


## 2. Create Lakebase Tables

In [9]:
# Configure your target catalog/schema for Genie360 metadata tables
GENIE360_CATALOG = "genie360"  # Change to your catalog
GENIE360_SCHEMA = "metadata"   # Change to your schema

In [10]:
# Create catalog and schema if they don't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {GENIE360_CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GENIE360_CATALOG}.{GENIE360_SCHEMA}")
print(f"✅ Catalog/Schema ready: {GENIE360_CATALOG}.{GENIE360_SCHEMA}")

✅ Catalog/Schema ready: genie360.metadata


In [13]:
# Table 1: genie_spaces — Tracks managed Genie Spaces
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GENIE360_CATALOG}.{GENIE360_SCHEMA}.genie_spaces (
    space_id STRING NOT NULL,
    space_name STRING,
    workspace_url STRING,
    owner STRING,
    last_analyzed TIMESTAMP,
    last_score DOUBLE,
    total_queries_analyzed INT,
    total_rewrites_injected INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Genie360: Managed Genie Space registry'
""")
print("✅ Table created: genie_spaces")

✅ Table created: genie_spaces


In [14]:
# Table 2: query_analysis — Anti-pattern detection results
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GENIE360_CATALOG}.{GENIE360_SCHEMA}.query_analysis (
    analysis_id STRING NOT NULL,
    space_id STRING NOT NULL,
    query_id STRING,
    original_sql STRING,
    sql_fingerprint STRING,
    anti_patterns STRING,
    max_severity STRING,
    total_issues INT,
    estimated_cost_usd DOUBLE,
    duration_tier STRING,
    volume_tier STRING,
    analyzed_at TIMESTAMP
)
USING DELTA
COMMENT 'Genie360: Query-level anti-pattern analysis results'
""")
print("✅ Table created: query_analysis")

✅ Table created: query_analysis


In [15]:
# Table 3: rewrite_history — SQL rewrite candidates and outcomes
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GENIE360_CATALOG}.{GENIE360_SCHEMA}.rewrite_history (
    rewrite_id STRING NOT NULL,
    space_id STRING NOT NULL,
    query_id STRING,
    original_sql STRING,
    rule_rewritten_sql STRING,
    llm_rewritten_sql STRING,
    final_sql STRING,
    rules_applied STRING,
    confidence_score DOUBLE,
    estimated_savings_usd DOUBLE,
    status STRING,
    created_at TIMESTAMP,
    injected_at TIMESTAMP
)
USING DELTA
COMMENT 'Genie360: SQL rewrite candidates with before/after and confidence'
""")
print("✅ Table created: rewrite_history")

✅ Table created: rewrite_history


In [16]:
# Table 4: injection_log — Audit trail for Genie Space updates
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GENIE360_CATALOG}.{GENIE360_SCHEMA}.injection_log (
    injection_id STRING NOT NULL,
    space_id STRING NOT NULL,
    rewrites_injected INT,
    rewrites_failed INT,
    previous_serialized_space STRING,
    new_serialized_space STRING,
    success BOOLEAN,
    error_message STRING,
    injected_at TIMESTAMP,
    rolled_back_at TIMESTAMP
)
USING DELTA
COMMENT 'Genie360: Injection audit trail with rollback support'
""")
print("✅ Table created: injection_log")

✅ Table created: injection_log


## 3. Validate All Tables

In [17]:
tables = ["genie_spaces", "query_analysis", "rewrite_history", "injection_log"]
for table in tables:
    fqn = f"{GENIE360_CATALOG}.{GENIE360_SCHEMA}.{table}"
    try:
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {fqn}").collect()[0]["cnt"]
        print(f"  ✅ {fqn} — {count} rows")
    except Exception as e:
        print(f"  ❌ {fqn} — {e}")

print("\n🎯 Infrastructure setup complete!")

  ✅ genie360.metadata.genie_spaces — 0 rows


  ✅ genie360.metadata.query_analysis — 0 rows


  ✅ genie360.metadata.rewrite_history — 0 rows


  ✅ genie360.metadata.injection_log — 0 rows

🎯 Infrastructure setup complete!
